In [2]:
# imports
import pandas as pd
import numpy as np
import re

In [3]:
# load mastersportal dataset
masters = pd.read_csv("/Users/thetsusann/Desktop/BIA/Project/Progress/code/datasets/mastersportal-programs.csv")
masters.head()

,country_name,country_code,university_name,university_rank,program_name,program_type,deadline,duration,language,tution_1_currency,...,tution_2_money,tution_2_type,tuition_price_specification,start_date,ielts_score,structure,academic_req,facts,city,program_url
0,Armenia,ARM,American University of Armenia,NaN,Economics,MSc,2004-07-18T00:00:00Z,NaN,English,EUR,...,2108.0,EU/EEA,Tuition (Year),2018-09-01 00:00:00,6.5,['Quantitative Methods for Economists (Mathema...,"<section id=""AcademicRequirements""> <h2>Academ...",['Starting in 2018-09-01 00:00:00 You can...,['Yerevan'],http://www.mastersportal.eu/studies/71101/econ...
1,Armenia,ARM,American University of Armenia,NaN,Political Science and International Affairs,Master,2031-07-18T00:00:00Z,24 months,English,EUR,...,2500.0,National,Tuition (Year),2018-08-22 00:00:00,6.5,NaN,"<section id=""AcademicRequirements""> <h2>Academ...",['Starting in 2018-08-22 00:00:00 You can...,['Yerevan'],http://www.mastersportal.eu/studies/71085/poli...
2,Armenia,ARM,American University of Armenia,NaN,Business Administration,MBA,2004-07-18T00:00:00Z,NaN,English,EUR,...,2499.0,EU/EEA,Tuition (Year),2018-09-01 00:00:00,6.5,['Managers with practical knowledge of account...,"<section id=""AcademicRequirements""> <h2>Academ...",['Starting in 2018-09-01 00:00:00 You can...,['Yerevan'],http://www.mastersportal.eu/studies/71102/busi...
3,Armenia,ARM,American University of Armenia,NaN,Computer and Information Science,MSc,NaN,24 months,English,EUR,...,2500.0,National,Tuition (Year),NaN,6.5,['Introduction to Object-Oriented Programming'...,"<section id=""AcademicRequirements""> <h2>Academ...",['Deadline and start date Application deadline...,['Yerevan'],http://www.mastersportal.eu/studies/71104/comp...
4,Armenia,ARM,American University of Armenia,NaN,Industrial Engineering and Systems Management,MEng,2031-07-18T00:00:00Z,24 months,English,EUR,...,2500.0,National,Tuition (Year),2018-08-22 00:00:00,6.5,"['Probability Theory', 'Analysis and Design of...","<section id=""AcademicRequirements""> <h2>Academ...",['Starting in 2018-08-22 00:00:00 You can...,['Yerevan'],http://www.mastersportal.eu/studies/71103/indu...


In [4]:
# Basics 
print("Shape:", masters.shape)
print("\nColumns:")
print(masters.columns.tolist())

Shape: (60425, 23)

Columns:
['country_name', 'country_code', 'university_name', 'university_rank', 'program_name', 'program_type', 'deadline', 'duration', 'language', 'tution_1_currency', 'tution_1_money', 'tution_1_type', 'tution_2_currency', 'tution_2_money', 'tution_2_type', 'tuition_price_specification', 'start_date', 'ielts_score', 'structure', 'academic_req', 'facts', 'city', 'program_url']


In [5]:
# Inspect program_name and program_type
masters[["program_name", "program_type"]].head(10) 

,program_name,program_type
0,Economics,MSc
1,Political Science and International Affairs,Master
2,Business Administration,MBA
3,Computer and Information Science,MSc
4,Industrial Engineering and Systems Management,MEng
5,Law,LLM
6,Public Health Program,Master
7,Master of Business Administration in Tourism a...,MBA
8,Aruban Law,LLM
9,Asset and Maintenance Management,Postgraduate Diploma


In [6]:
# Check for missing values
masters[["program_name", "program_type"]].isnull().sum()

program_name    0
program_type    0
dtype: int64

In [12]:
# Inspect unique program_type 
masters["program_type"].nunique()

502

In [22]:
# Inspect all program_type distribution and save to CSV
program_type_counts = masters["program_type"].value_counts().reset_index()
program_type_counts.columns = ["program_type", "count"]
program_type_counts.to_csv("/Users/thetsusann/Desktop/BIA/Project/Progress/code/datasets/program_type_counts.csv", index=False)
program_type_counts

,program_type,count
0,MSc,24715
1,MA,13017
2,Master,10008
3,MBA,2677
4,MEd,1697
...,...,...
497,MHCA,1
498,MSHA,1
499,MMSM,1
500,JDMSW,1


In [23]:
# Remove program_type distribution < 10
valid_program_types = program_type_counts[program_type_counts["count"] >= 10]["program_type"]
masters = masters[masters["program_type"].isin(valid_program_types)].copy()
print("Shape after filtering program_type:", masters.shape)

Shape after filtering program_type: (59603, 23)


In [24]:
# Remove rows with missing program_name
masters = masters.dropna(subset=["program_name"]).copy()
print("Shape after dropping missing program_name:", masters.shape)

Shape after dropping missing program_name: (59603, 23)


In [26]:
# Remove non-master pograms
exclude_types = [
    "Postgraduate Certificate",
    "Postgraduate Diploma",
    "Preparation Course",
    "Specialization",
    "Graduate Certificate",
    "MScPGDipPGCert"
]

masters = masters[~masters["program_type"].isin(exclude_types)].copy()
print("Shape after removal", masters.shape)
print("\nRemaining program_type values:")
print(masters["program_type"].value_counts(dropna=False).head(30))

Shape after removal (57126, 23)

Remaining program_type values:
program_type
MSc                    24715
MA                     13017
Master                 10008
MBA                     2677
MEd                     1697
LLM                     1053
MEng                     987
MPhil                    692
MRes                     584
Pre-Master               462
MFA                      185
MM                       116
MLitt                    102
MPA                       95
MPH                       77
MAT                       63
MSW                       62
Master of Music           51
MSN                       36
Master of Fine Arts       32
MSEd                      30
MArch                     29
MPhys                     28
MAcc                      22
MSci                      22
MDiv                      21
MMus                      17
MSS                       17
MChem                     17
MPS                       17
Name: count, dtype: int64


In [27]:
# Keep only relevant columns
masters_clean = masters[["program_name", "program_type"]].copy()
masters_clean.head()

,program_name,program_type
0,Economics,MSc
1,Political Science and International Affairs,Master
2,Business Administration,MBA
3,Computer and Information Science,MSc
4,Industrial Engineering and Systems Management,MEng


In [ ]:
# lowercase + strip whitespace
masters_clean["program_name_clean"] = (
    masters_clean["program_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

masters_clean.head(10)

,program_name,program_type,program_name_clean
0,Economics,MSc,economics
1,Political Science and International Affairs,Master,political science and international affairs
2,Business Administration,MBA,business administration
3,Computer and Information Science,MSc,computer and information science
4,Industrial Engineering and Systems Management,MEng,industrial engineering and systems management
5,Law,LLM,law
6,Public Health Program,Master,public health program
7,Master of Business Administration in Tourism a...,MBA,master of business administration in tourism a...
8,Aruban Law,LLM,aruban law
16,Human Resource Management,Master,human resource management


In [ ]:
# Define cleaning function
def clean_degree_name(text):
    text = str(text).lower().strip()
    
    # remove punctuation-like separators
    text = re.sub(r"[&/,+()\-]", " ", text)
    
    # remove common degree words / abbreviations
    patterns_to_remove = [
        r"\bmaster of\b",
        r"\bmasters of\b",
        r"\bmaster\b",
        r"\bmasters\b",
        r"\bmsc\b",
        r"\bms\b",
        r"\bma\b",
        r"\bmba\b",
        r"\bmeng\b",
        r"\bllm\b",
        r"\bmfa\b",
        r"\bmed\b",
        r"\bmarch\b",
        r"\bmph\b",
        r"\bdegree\b",
        r"\bprogramme\b",
        r"\bprogram\b"
    ]
    
    for pattern in patterns_to_remove:
        text = re.sub(pattern, " ", text)
    
    # collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [30]:
# Apply cleaning function
masters_clean["degree_text"] = masters_clean["program_name_clean"].apply(clean_degree_name)
masters_clean[["program_name", "program_type", "degree_text"]].head(20)

,program_name,program_type,degree_text
0,Economics,MSc,economics
1,Political Science and International Affairs,Master,political science and international affairs
2,Business Administration,MBA,business administration
3,Computer and Information Science,MSc,computer and information science
4,Industrial Engineering and Systems Management,MEng,industrial engineering and systems management
5,Law,LLM,law
6,Public Health Program,Master,public health
7,Master of Business Administration in Tourism a...,MBA,business administration in tourism and interna...
8,Aruban Law,LLM,aruban law
16,Human Resource Management,Master,human resource management


In [31]:
# Inspect random samples
masters_clean[["program_name", "program_type", "degree_text"]].sample(20, random_state=42)

,program_name,program_type,degree_text
37008,Environmental Engineering,MSc,environmental engineering
13437,Strategic Communication,MSc,strategic communication
6585,Animal Science,MSc,animal science
29060,Environment and Development,MSc,environment and development
50614,Molecular Biology,MSc,molecular biology
3530,Criminology (MA,MA,criminology
5089,Electrical and Computer Engineering,MEng,electrical and computer engineering
3378,Comparative Vertebrate Morphology,MSc,comparative vertebrate morphology
16795,Cross-Cultural Psychology,MSc,cross cultural psychology
56424,Public Administration,Master,public administration


In [ ]:
# Remove empty cleaned values
masters_clean["degree_text"] = masters_clean["degree_text"].replace("", np.nan)
masters_clean = masters_clean.dropna(subset=["degree_text"]).copy()

print("Shape after removing empty degree_text:", masters_clean.shape)

Shape after removing empty degree_text: (57112, 4)


In [40]:
# Inspect most common cleaned degree_text values
masters_clean["degree_text"].value_counts().head(50)

degree_text
business administration              754
computer science                     428
chemistry                            354
history                              353
economics                            350
mathematics                          333
mechanical engineering               331
finance                              313
english                              292
social work                          276
physics                              268
biology                              266
management                           259
civil engineering                    247
public health                        245
psychology                           244
accounting                           239
public administration                230
education                            212
electrical engineering               207
architecture                         198
philosophy                           192
nursing                              191
sociology                            186
chem

In [ ]:
# # Save cleaned dataset v1
# masters_clean.to_csv("datasets/masters_cleaned_for_mapping_v1.csv", index=False)
# print("Saved: datasets/masters_cleaned_for_mapping_v1.csv")

Saved: masters_cleaned_for_mapping.csv


In [ ]:
# Inspect suspicious values
suspicious = masters_clean[
    masters_clean["degree_text"].str.len() <= 4
][["program_name", "program_type", "degree_text"]]

with pd.option_context("display.max_rows", None):
    display(suspicious)

In [46]:
# Count short labels
short_labels = (
    masters_clean["degree_text"]
    .value_counts()
    .reset_index()
)
short_labels.columns = ["degree_text", "count"]

with pd.option_context("display.max_rows", None):
    display(short_labels[short_labels["degree_text"].str.len() <= 5])

,degree_text,count
27,music,172
40,law,116
78,art,70
86,laws,59
206,tesol,27
227,dance,25
246,media,23
271,arts,21
274,film,21
486,drama,12


In [ ]:
# Inspect suspicious labels in context
suspect_terms = [
    # real but short
    "law", "laws", "art", "arts", "film", "media",

    # weak / vague
    "child", "work", "mind", "life", "cell",

    # garbage
    "one", "usa", "uk", "pre", "ace", "imba", "acca", "msx", "iese", "flex",

    # abbreviations 
    "jd", "md", "bme", "msw", "mpa", "mmus", "cad"
]

masters_clean[
    masters_clean["degree_text"].isin(suspect_terms)
][["program_name", "program_type", "degree_text"]].sort_values("degree_text").head(200)

,program_name,program_type,degree_text
12166,ACCA Programme,MBA,acca
19327,ACE,Master,ace
8464,ACE,Master,ace
3487,ACE,Master,ace
13439,ACE,Master,ace
...,...,...,...
19183,Law,LLM,law
23366,Law,MA,law
12371,Law,LLM,law
12824,Law,LLM,law


In [48]:
normalize_map = {
    "laws": "law",
    "arts": "art",
    "math": "mathematics",
    "it": "information technology",
    "accountancy": "accounting",
    "executive business administration": "business administration",
    "computer and information science": "computer science",
    "computer information science": "computer science"
}

remove_terms = {
    "one", "usa", "uk", "pre", "ace", "imba", "acca", "msx", "iese", "flex", "jd", "md", "bme", "msw", "mpa", "mmus", "cad"
}

In [49]:
masters_clean["degree_norm"] = masters_clean["degree_text"].replace(normalize_map)

masters_clean[
    masters_clean["degree_text"] != masters_clean["degree_norm"]
][["program_name", "program_type", "degree_text", "degree_norm"]].head(100)

,program_name,program_type,degree_text,degree_norm
3,Computer and Information Science,MSc,computer and information science,computer science
246,Laws,Master,laws,law
419,Arts,Master,arts,art
439,Laws,Master,laws,law
639,Arts,Master,arts,art
...,...,...,...,...
21374,Executive Business Administration,MBA,executive business administration,business administration
21774,MBA - Executive Master of Business Administration,MBA,executive business administration,business administration
21794,MBA - Executive Master of Business Administration,MBA,executive business administration,business administration
22012,MBA - Executive Master of Business Administration,MBA,executive business administration,business administration


In [51]:
masters_clean["remove_flag"] = masters_clean["degree_norm"].isin(remove_terms)

masters_clean[["program_name", "program_type", "degree_text", "degree_norm", "remove_flag"]].sample(30, random_state=42)

,program_name,program_type,degree_text,degree_norm,remove_flag
36883,Chemical Engineering,MSc,chemical engineering,chemical engineering,False
32158,Music Theatre,MA,music theatre,music theatre,False
23509,Philosophy,MRes,philosophy,philosophy,False
33835,Marketing,MSc,marketing,marketing,False
38721,Writing For Children,Master,writing for children,writing for children,False
58335,Logistics,MSc,logistics,logistics,False
48901,International Master of Business Administratio...,MBA,international business administration in foren...,international business administration in foren...,False
24176,Social Research Methods,MSc,social research methods,social research methods,False
58046,Environmental Science,MSc,environmental science,environmental science,False
53061,Thanatology,MA,thanatology,thanatology,False


In [52]:
print("Rows flagged for removal:", masters_clean["remove_flag"].sum())
print("Total rows:", len(masters_clean))

Rows flagged for removal: 27
Total rows: 57112


In [53]:
# REMOVE flagged rows and save final cleaned dataset
masters_clean = masters_clean[~masters_clean["remove_flag"]].copy()

print("Final shape after removal:", masters_clean.shape)

Final shape after removal: (57085, 6)


In [ ]:
# Save cleaned dataset
masters_clean.to_csv("datasets/masters_cleaned_for_mapping.csv", index=False)
print("Saved: datasets/masters_cleaned_for_mapping.csv")

Saved: datasets/masters_cleaned_for_mapping_final.csv


In [61]:
import openpyxl

# Load occupation dataset
jobs = pd.read_excel("/Users/thetsusann/Desktop/BIA/Project/Progress/code/datasets/occupation-salary.xlsx", engine="openpyxl")
jobs.head()

,OCC_CODE,OCC_TITLE,OCC_GROUP,TOT_EMP,EMP_PRSE,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,00-0000,All Occupations,total,140400040,0.1,23.86,49630,0.1,9.27,11.6,17.81,28.92,45.45,19290,24140,37040,60150,94540,NaN,NaN
1,11-0000,Management Occupations,major,7090790,0.2,56.74,118020,0.1,22.76,32.99,48.46,70.72,#,47330,68630,100790,147090,#,NaN,NaN
2,11-1000,Top Executives,minor,2465800,0.2,61.03,126950,0.2,20.58,31.45,49.19,78.35,#,42810,65420,102320,162970,#,NaN,NaN
3,11-1010,Chief Executives,broad,223260,0.7,93.44,194350,0.4,33.55,54.86,87.12,#,#,69780,114100,181210,#,#,NaN,NaN
4,11-1011,Chief Executives,detailed,223260,0.7,93.44,194350,0.4,33.55,54.86,87.12,#,#,69780,114100,181210,#,#,NaN,NaN


In [62]:
print("Shape:", jobs.shape)
print("\nColumns:")
print(jobs.columns.tolist())

Shape: (1394, 20)

Columns:
['OCC_CODE', 'OCC_TITLE', 'OCC_GROUP', 'TOT_EMP', 'EMP_PRSE', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']


In [64]:
jobs[["OCC_TITLE"]].head(10)

,OCC_TITLE
0,All Occupations
1,Management Occupations
2,Top Executives
3,Chief Executives
4,Chief Executives
5,General and Operations Managers
6,General and Operations Managers
7,Legislators
8,Legislators
9,"Advertising, Marketing, Promotions, Public Rel..."


In [ ]:
jobs[["OCC_TITLE"]].isnull().sum() # Check missing values

OCC_TITLE    0
dtype: int64

In [66]:
print("Unique OCC_TITLE:", jobs["OCC_TITLE"].nunique())

Unique OCC_TITLE: 1121


In [67]:
jobs_clean = jobs[["OCC_TITLE"]].copy()
jobs_clean = jobs_clean.dropna(subset=["OCC_TITLE"]).copy()
jobs_clean.head()

,OCC_TITLE
0,All Occupations
1,Management Occupations
2,Top Executives
3,Chief Executives
4,Chief Executives


In [69]:
# Remove the row with "All Occupations"
jobs_clean = jobs_clean[jobs_clean["OCC_TITLE"] != "All Occupations"]
jobs_clean.head()

,OCC_TITLE
1,Management Occupations
2,Top Executives
3,Chief Executives
4,Chief Executives
5,General and Operations Managers


In [70]:
jobs_clean["occupation_text"] = (
    jobs_clean["OCC_TITLE"]
    .astype(str)
    .str.lower()
    .str.strip()
)

jobs_clean.head(20)

,OCC_TITLE,occupation_text
1,Management Occupations,management occupations
2,Top Executives,top executives
3,Chief Executives,chief executives
4,Chief Executives,chief executives
5,General and Operations Managers,general and operations managers
6,General and Operations Managers,general and operations managers
7,Legislators,legislators
8,Legislators,legislators
9,"Advertising, Marketing, Promotions, Public Rel...","advertising, marketing, promotions, public rel..."
10,Advertising and Promotions Managers,advertising and promotions managers


In [71]:
jobs_clean = jobs_clean.drop_duplicates(subset=["occupation_text"]).copy()
print("Unique cleaned occupations:", jobs_clean.shape[0])
jobs_clean.head(20)

Unique cleaned occupations: 1120


,OCC_TITLE,occupation_text
1,Management Occupations,management occupations
2,Top Executives,top executives
3,Chief Executives,chief executives
5,General and Operations Managers,general and operations managers
7,Legislators,legislators
9,"Advertising, Marketing, Promotions, Public Rel...","advertising, marketing, promotions, public rel..."
10,Advertising and Promotions Managers,advertising and promotions managers
12,Marketing and Sales Managers,marketing and sales managers
13,Marketing Managers,marketing managers
14,Sales Managers,sales managers


In [72]:
jobs_clean.sample(30, random_state=42)

,OCC_TITLE,occupation_text
318,"Religious Workers, All Other","religious workers, all other"
145,Computer User Support Specialists,computer user support specialists
1198,"Metal Workers and Plastic Workers, All Other","metal workers and plastic workers, all other"
1315,"Ambulance Drivers and Attendants, Except Emerg...","ambulance drivers and attendants, except emerg..."
630,Detectives and Criminal Investigators,detectives and criminal investigators
1168,"Rolling Machine Setters, Operators, and Tender...","rolling machine setters, operators, and tender..."
804,Telemarketers,telemarketers
342,"Engineering and Architecture Teachers, Postsec...","engineering and architecture teachers, postsec..."
182,Electrical and Electronics Engineers,electrical and electronics engineers
373,"Arts, Communications, and Humanities Teachers,...","arts, communications, and humanities teachers,..."


In [73]:
# Clean more aggressively by removing punctuation and the word "All Other"
def clean_occupation(text):
    text = str(text).lower().strip()
    
    # remove punctuation-like separators
    text = re.sub(r"[&/,+()\-]", " ", text)
    
    # remove "all other"
    text = re.sub(r"\ball other\b", " ", text)
    
    # collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [74]:
# Aplly cleaning function
jobs_clean["occupation_clean"] = jobs_clean["occupation_text"].apply(clean_occupation)
jobs_clean[["OCC_TITLE", "occupation_text", "occupation_clean"]].head(20)

,OCC_TITLE,occupation_text,occupation_clean
1,Management Occupations,management occupations,management occupations
2,Top Executives,top executives,top executives
3,Chief Executives,chief executives,chief executives
5,General and Operations Managers,general and operations managers,general and operations managers
7,Legislators,legislators,legislators
9,"Advertising, Marketing, Promotions, Public Rel...","advertising, marketing, promotions, public rel...",advertising marketing promotions public relati...
10,Advertising and Promotions Managers,advertising and promotions managers,advertising and promotions managers
12,Marketing and Sales Managers,marketing and sales managers,marketing and sales managers
13,Marketing Managers,marketing managers,marketing managers
14,Sales Managers,sales managers,sales managers


In [75]:
jobs_clean.sample(30, random_state=42)

,OCC_TITLE,occupation_text,occupation_clean
318,"Religious Workers, All Other","religious workers, all other",religious workers
145,Computer User Support Specialists,computer user support specialists,computer user support specialists
1198,"Metal Workers and Plastic Workers, All Other","metal workers and plastic workers, all other",metal workers and plastic workers
1315,"Ambulance Drivers and Attendants, Except Emerg...","ambulance drivers and attendants, except emerg...",ambulance drivers and attendants except emerge...
630,Detectives and Criminal Investigators,detectives and criminal investigators,detectives and criminal investigators
1168,"Rolling Machine Setters, Operators, and Tender...","rolling machine setters, operators, and tender...",rolling machine setters operators and tenders ...
804,Telemarketers,telemarketers,telemarketers
342,"Engineering and Architecture Teachers, Postsec...","engineering and architecture teachers, postsec...",engineering and architecture teachers postseco...
182,Electrical and Electronics Engineers,electrical and electronics engineers,electrical and electronics engineers
373,"Arts, Communications, and Humanities Teachers,...","arts, communications, and humanities teachers,...",arts communications and humanities teachers po...


In [76]:
jobs_clean.to_csv("datasets/occupations_cleaned_for_mapping.csv", index=False)
print("Saved: datasets/occupations_cleaned_for_mapping.csv")

Saved: datasets/occupations_cleaned_for_mapping.csv
